# 부록 10. LangSmith로 Tracing과 디버깅

LangSmith는 LangChain/LangGraph 애플리케이션의 실행을 추적, 모니터링, 디버깅할 수 있는 플랫폼입니다.

## 학습 목표
- LangSmith Tracing 설정 방법
- 실행 추적 및 분석
- LangGraph Studio 사용법

## 1. LangSmith 소개

### LangSmith란?
- LLM 애플리케이션의 **관찰 가능성(Observability)** 플랫폼
- 실행 추적, 디버깅, 평가, 모니터링 기능 제공

### 주요 기능
| 기능 | 설명 |
|------|------|
| Tracing | 모든 LLM 호출 및 체인 실행 추적 |
| Debugging | 오류 추적 및 단계별 분석 |
| Evaluation | 에이전트 성능 평가 |
| Monitoring | 프로덕션 성능 모니터링 |

## 2. 환경 설정

### 필요한 환경 변수

```bash
# .env 파일에 추가
LANGSMITH_API_KEY=lsv2_...        # LangSmith API 키
LANGSMITH_TRACING=true            # Tracing 활성화
LANGSMITH_PROJECT=my-project      # 프로젝트 이름 (선택)
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# 환경 변수 확인
print("=== LangSmith 설정 상태 ===")
print(f"LANGSMITH_API_KEY: {'설정됨' if os.getenv('LANGSMITH_API_KEY') else '미설정 ⚠️'}")
print(f"LANGSMITH_TRACING: {os.getenv('LANGSMITH_TRACING', '미설정')}")
print(f"LANGSMITH_PROJECT: {os.getenv('LANGSMITH_PROJECT', '미설정 (default 사용)')}")
print(f"OPENAI_API_KEY: {'설정됨' if os.getenv('OPENAI_API_KEY') else '미설정 ⚠️'}")

In [ ]:
# Tracing 활성화 (환경 변수로 설정하는 것이 권장됨)
os.environ["LANGSMITH_TRACING"] = "true"

# 프로젝트 이름 설정 (선택)
os.environ["LANGSMITH_PROJECT"] = "fc-agent-bible-appendix"

print("Tracing이 활성화되었습니다.")

## 3. 기본 Tracing 사용

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def calculate(expression: str) -> str:
    """수학 계산을 수행합니다."""
    try:
        return str(eval(expression))
    except:
        return "계산 오류"

@tool
def get_weather(city: str) -> str:
    """날씨를 조회합니다."""
    return f"{city}: 맑음, 22°C"

# 에이전트 생성 - LANGSMITH_TRACING=true면 자동으로 추적됨
agent = create_agent(
    model="gpt-4o-mini",
    tools=[calculate, get_weather],
    system_prompt="당신은 도움이 되는 어시스턴트입니다."
)

print("에이전트가 생성되었습니다. 실행이 LangSmith에서 추적됩니다.")

In [ ]:
# 에이전트 실행 - 자동으로 LangSmith에 기록됨
result = agent.invoke({
    "messages": [{"role": "user", "content": "123 * 456은 얼마야?"}]
})

print(f"응답: {result['messages'][-1].content}")
print("\n이 실행은 LangSmith에서 확인할 수 있습니다:")
print("https://smith.langchain.com")

## 4. 메타데이터와 태그 추가

실행에 추가 정보를 첨부하여 나중에 필터링하거나 분석할 수 있습니다.

In [ ]:
# config로 메타데이터와 태그 추가
config = {
    "metadata": {
        "user_id": "user-123",
        "session_id": "session-456",
        "environment": "development"
    },
    "tags": ["test", "weather-query", "appendix-demo"]
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 알려줘"}]},
    config=config
)

print(f"응답: {result['messages'][-1].content}")
print("\nLangSmith에서 태그로 필터링 가능: test, weather-query, appendix-demo")

## 5. 실행 이름 지정

실행에 이름을 지정하여 LangSmith에서 쉽게 찾을 수 있습니다.

In [ ]:
# run_name으로 실행 이름 지정
config_with_name = {
    "run_name": "날씨-계산-복합-쿼리",
    "metadata": {"query_type": "complex"}
}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨랑 15*20 계산해줘"}]},
    config=config_with_name
)

print(f"응답: {result['messages'][-1].content}")
print("\nLangSmith에서 '날씨-계산-복합-쿼리'로 검색 가능")

## 6. 선택적 Tracing

특정 실행만 추적하고 싶을 때 사용합니다.

In [ ]:
from langchain_core.tracers.context import tracing_v2_enabled

# 전역 tracing 비활성화
os.environ["LANGSMITH_TRACING"] = "false"

# 이 실행은 추적되지 않음
result1 = agent.invoke({
    "messages": [{"role": "user", "content": "1+1"}]
})
print("실행 1 (추적 안됨): 완료")

# 컨텍스트 매니저로 선택적 추적
with tracing_v2_enabled(project_name="selective-tracing"):
    result2 = agent.invoke({
        "messages": [{"role": "user", "content": "2+2"}]
    })
    print("실행 2 (추적됨): 완료")

# 다시 전역 tracing 활성화
os.environ["LANGSMITH_TRACING"] = "true"

## 7. LangGraph Studio 소개

### LangGraph Studio란?
- LangGraph 그래프를 시각적으로 탐색하고 디버깅할 수 있는 도구
- 로컬 개발 서버 또는 배포된 앱에 연결

### 로컬에서 Studio 사용하기

```bash
# LangGraph CLI 설치
pip install langgraph-cli

# 개발 서버 시작 (langgraph.json 필요)
langgraph dev

# Studio 접속
# https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
```

In [ ]:
# langgraph.json 예시
langgraph_config = {
    "dependencies": ["."],
    "graphs": {
        "agent": "./agent.py:graph"
    },
    "env": ".env"
}

import json
print("langgraph.json 예시:")
print(json.dumps(langgraph_config, indent=2))

## 8. LangSmith 대시보드 활용

### 주요 기능

1. **Traces 탭**
   - 모든 실행 목록 확인
   - 필터링 (태그, 날짜, 상태)
   - 각 실행의 상세 정보

2. **실행 상세 보기**
   - 입력/출력 확인
   - 각 단계별 실행 시간
   - 토큰 사용량
   - 에러 추적

3. **비교 기능**
   - 여러 실행 비교
   - A/B 테스트

4. **모니터링**
   - 대시보드 생성
   - 알림 설정

In [ ]:
# 다양한 실행을 생성하여 대시보드에서 확인
test_queries = [
    "안녕하세요!",
    "100 + 200 계산해줘",
    "도쿄 날씨 알려줘",
    "서울 날씨와 50 * 30 계산해줘"
]

for i, query in enumerate(test_queries):
    config = {
        "run_name": f"test-run-{i+1}",
        "tags": ["batch-test", f"query-{i+1}"]
    }
    
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config=config
    )
    
    print(f"✅ Query {i+1} 완료: {query[:30]}...")

print("\n모든 실행이 LangSmith에 기록되었습니다.")
print("https://smith.langchain.com 에서 'batch-test' 태그로 필터링하여 확인하세요.")

## 9. 종료 처리

서버리스 환경에서는 프로세스 종료 전에 모든 트레이스가 전송되도록 해야 합니다.

In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

# 마지막 실행
result = agent.invoke({
    "messages": [{"role": "user", "content": "마지막 테스트"}]
})

# 모든 트레이스가 전송될 때까지 대기
wait_for_all_tracers()

print("모든 트레이스가 전송되었습니다.")

## 10. 요약

### 이 노트북에서 배운 것

1. **LangSmith 설정**
   - `LANGSMITH_API_KEY`: API 키
   - `LANGSMITH_TRACING=true`: 추적 활성화
   - `LANGSMITH_PROJECT`: 프로젝트 이름

2. **실행 추적**
   - 자동 추적 (환경 변수 설정 시)
   - 메타데이터와 태그 추가
   - 실행 이름 지정

3. **선택적 추적**
   - `tracing_v2_enabled` 컨텍스트 매니저

4. **LangGraph Studio**
   - 시각적 그래프 탐색
   - 로컬 개발 서버 연결

5. **대시보드 활용**
   - 실행 목록 및 필터링
   - 상세 분석
   - 모니터링

### 부록 시리즈 완료!

이 부록 시리즈에서 다룬 내용:
1. LangChain과 LangGraph 소개
2. init_chat_model로 모델 초기화
3. Tool 데코레이터와 ToolRuntime
4. create_agent로 에이전트 만들기
5. StateGraph API로 커스텀 그래프 구축
6. Middleware로 실행 제어
7. Memory와 State 관리
8. Structured Output
9. Streaming
10. LangSmith Tracing